## Quantum Sand Project Template Reqwest 001

### Import dependencies

We will load the external crate dependencies `reqwest` and `tokio`.

In [1]:
:dep reqwest = { version = "0.13.4" }
:dep tokio = { version = "1.52.3", features = ["full"] }

### Running Quantum Sand Rails locally

If you are just looking at this notebook statically in a browser or VS Code you can just read this notebook without running the Rust code.

Having the Quantum Sand Rails; Ruby-on-Rails PostGIS API; running locally is necessary for running the Rust code later in this notebook.

The `seeds.rb` within Quantum Sand Rails has some test data which we can use for this example;

This block of Ruby code imports some seed data from `geospatial_traces = []` into the `geospatial_traces` table;

```ruby
geospatial_traces = [
  {
    name: "Harbour seals in Christchurch Bay",
    geospatial: "POINT(50.713294760937224 -1.6601009519012668)",
    data: {
      name: "Harbour seal",
      scientific_name: "Phoca vitulina"
    }
  }
]

geospatial_traces.each do |geospatial_trace|
  GeospatialTrace.where(name: geospatial_trace[:name]).first_or_create do |trace|
    trace.name = geospatial_trace[:name]
    trace.geospatial = geospatial_trace[:geospatial]
    trace.data = geospatial_trace[:data]

    puts "Created geospatial_trace with name: #{geospatial_trace[:name]}"
  end
end
```

The `geospatial_traces` have the following attributes; the columns in the database table;

* `name` is a string, `"Harbour seals in Christchurch Bay"`.
* `geospatial` is PostGIS geometry, `"POINT(50.713294760937224 -1.6601009519012668)"`.
* `data` is JSON, `{name: "Harbour seal", scientific_name: "Phoca vitulina"}.`

We will use Rust with Reqwest to `get()` this data from the `geospatial_traces` database table within the Rails app.

### Access the Rails PostGIS API over HTTP using Reqwest

If you are just looking at this notebook statically in a browser or VS Code you can just read the outcome of this section.

You need to run Quantum Sand Rails locally if you want to run this Rust code.

`#[tokio::main]` marks the async function to be executed by the selected runtime.

We can define the `main()` function like so;

```rust
#[tokio::main]
async fn main() -> Result<(), Box<dyn std::error::Error>> {
    let response = reqwest::get("http://127.0.0.1:3000/geospatial_traces")
        .await?
        .text()
        .await?;
    println!("{response:#?}");
    Ok(())
}
```

Using `get()` within Reqwest with the argument;

* `url` is `"http://127.0.0.1:3000/geospatial_traces"`.

Sends a GET request and returns the response body into `response`.

Note that `.await?` corresponds to the Rust keyword `await`, which suspends execution until the result of a `Future` is ready.

Also note that `.text()` within Reqwest gets the full response text.

Finally, we can print out the `response` using `println!()`.

Within Rust, `Ok` is one of two variants of the `Result` enum. It represents success or a successful operation.

The line `main()` will call this function.

In [2]:
#[tokio::main]
async fn main() -> Result<(), Box<dyn std::error::Error>> {
    let response = reqwest::get("http://127.0.0.1:3000/geospatial_traces")
        .await?
        .text()
        .await?;
    println!("{response:#?}");
    Ok(())
}

In [3]:
main()

"[{\"id\":1,\"name\":\"Harbour seals in Christchurch Bay\",\"geospatial\":\"POINT (50.713294760937224 -1.6601009519012668)\",\"data\":{\"name\":\"Harbour seal\",\"scientific_name\":\"Phoca vitulina\"},\"created_at\":\"2026-06-19T15:04:34.602Z\",\"updated_at\":\"2026-06-19T15:04:34.602Z\"}]"


Ok(())

You can see the same data that was used to seed the `geospatial_traces` table.

More to follow...